### Neural Network to Predict a Successful Vaginal Birth After C-Section

Now we'll test a neural network to see if it achieves better accuracy.

In [ ]:
from pickle import load

x_train_unfiltered = load(
    open("../../../data/processed/vbac/X_train_prepared.pkl", "rb")
)
x_test_unfiltered = load(open("../../../data/processed/vbac/X_test_prepared.pkl", "rb"))
y_test = load(open("../../../data/processed/vbac/y_test.pkl", "rb"))
y_train = load(open("../../../data/processed/vbac/y_train.pkl", "rb"))

feature_importances_df = load(
    open("../../../data/processed/vbac/feature_importances_df.pkl", "rb")
)

We'll use our `feature_importances` dataframe to train a few versions of the model with different configurations of the data.

First, we'll try training using any feature with an importance > 0.01.

In [ ]:
x_train_001 = x_train_unfiltered[
    :,
    feature_importances_df.loc[
        feature_importances_df["importance"] > 0.01
    ].index.tolist(),
]
x_test_001 = x_test_unfiltered[
    :,
    feature_importances_df.loc[
        feature_importances_df["importance"] > 0.01
    ].index.tolist(),
]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

nn_model_001 = Sequential()
nn_model_001.add(Dense(64, input_dim=x_train_001.shape[1], activation="relu"))
nn_model_001.add(Dense(32, activation="relu"))
nn_model_001.add(Dense(1, activation="sigmoid"))

nn_model_001.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

nn_model_001.fit(
    x_train_001,
    y_train,
    epochs=10,
    batch_size=100,
    validation_data=(x_test_001, y_test),
)

In [ ]:
predictions_001 = nn_model_001.predict(x_test_001)
predictions_001 = (predictions_001 > 0.5).astype(int)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, predictions_001)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot()
plt.show()

In [ ]:
from sklearn.metrics import f1_score

f1_001 = f1_score(y_test, predictions_001)

f1_001

Next we'll train a model with features of importance > 0.15.

In [ ]:
x_train_0015 = x_train_unfiltered[
    :,
    feature_importances_df.loc[
        feature_importances_df["importance"] > 0.015
    ].index.tolist(),
]
x_test_0015 = x_test_unfiltered[
    :,
    feature_importances_df.loc[
        feature_importances_df["importance"] > 0.015
    ].index.tolist(),
]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

nn_model_0015 = Sequential()
nn_model_0015.add(Dense(64, input_dim=x_train_0015.shape[1], activation="relu"))
nn_model_0015.add(Dense(32, activation="relu"))
nn_model_0015.add(Dense(1, activation="sigmoid"))

nn_model_0015.compile(
    loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"]
)

nn_model_0015.fit(
    x_train_0015,
    y_train,
    epochs=10,
    batch_size=100,
    validation_data=(x_test_0015, y_test),
)

In [ ]:
predictions_0015 = nn_model_0015.predict(x_test_0015)
predictions_0015 = (predictions_0015 > 0.5).astype(int)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, predictions_0015)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot()
plt.show()

In [ ]:
from sklearn.metrics import f1_score

f1_0015 = f1_score(y_test, predictions_0015)

f1_0015

Next we'll train a model with features of importance > 0.2.

In [ ]:
x_train_002 = x_train_unfiltered[
    :,
    feature_importances_df.loc[
        feature_importances_df["importance"] > 0.02
    ].index.tolist(),
]
x_test_002 = x_test_unfiltered[
    :,
    feature_importances_df.loc[
        feature_importances_df["importance"] > 0.02
    ].index.tolist(),
]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

nn_model_002 = Sequential()
nn_model_002.add(Dense(64, input_dim=x_train_002.shape[1], activation="relu"))
nn_model_002.add(Dense(32, activation="relu"))
nn_model_002.add(Dense(1, activation="sigmoid"))

nn_model_002.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

nn_model_002.fit(
    x_train_002,
    y_train,
    epochs=10,
    batch_size=100,
    validation_data=(x_test_002, y_test),
)

In [ ]:
predictions_002 = nn_model_002.predict(x_test_002)
predictions_002 = (predictions_002 > 0.5).astype(int)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, predictions_002)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot()
plt.show()

In [ ]:
from sklearn.metrics import f1_score

f1_002 = f1_score(y_test, predictions_002)

f1_002

Next we'll train a model with a subset of the features based on domain knowledge.

In [ ]:
custom_filtered_features_df = feature_importances_df.loc[
    feature_importances_df["importance"] > 0.01
].copy()

custom_filtered_features_df

Some of these features are duplicates of each other, or otherwise highly correlated. First, we'll clean those up.

In [ ]:
custom_filtered_features_df = custom_filtered_features_df.drop([1597, 1601])

Although paternal age plays a factor in fertility and pregnancy, we don't believe it to play a factor in VBAC. We'll rely on the mother's age instead of both ages.

In [ ]:
custom_filtered_features_df = custom_filtered_features_df.drop([1587])

We can now train our model.

In [ ]:
x_train_custom = x_train_unfiltered[
    :,
    custom_filtered_features_df.index.tolist(),
]
x_test_custom = x_test_unfiltered[
    :,
    custom_filtered_features_df.index.tolist(),
]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

nn_model_custom = Sequential()
nn_model_custom.add(Dense(64, input_dim=x_train_custom.shape[1], activation="relu"))
nn_model_custom.add(Dense(32, activation="relu"))
nn_model_custom.add(Dense(1, activation="sigmoid"))

nn_model_custom.compile(
    loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"]
)

nn_model_custom.fit(
    x_train_custom,
    y_train,
    epochs=10,
    batch_size=100,
    validation_data=(x_test_custom, y_test),
)

In [ ]:
predictions_custom = nn_model_custom.predict(x_test_custom)
predictions_custom = (predictions_custom > 0.5).astype(int)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, predictions_custom)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot()
plt.show()

In [ ]:
from sklearn.metrics import f1_score

f1_custom = f1_score(y_test, predictions_custom)

f1_custom

### Findings

In [ ]:
print("Importance 0.01 F1 score: ", f1_001)
print("Importance 0.015 F1 score: ", f1_0015)
print("Importance 0.02 F1 score: ", f1_002)
print("Custom filter F1 score: ", f1_custom)

The 0.01 and custom models performed at approximately the same level. We will move forward with the custom model, as its smaller feature set will be easier to implement in an interactive format.

In [ ]:
nn_model_custom.save("../../../models/vbac/nn_model.keras")

### Conclusion

Our neural network is able to predict VBAC outcomes with moderate accuracy, but it does not perform significantly better than our random forest classifier.